# Try to read tensorboard events from the directory

In [1]:
import polars as pl
from deeptan.utils.uni import collect_tensorboard_events

/home/wuch/miniforge3/envs/sc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# _path = "/mnt/hdd2/homext/wuch/xn2p/run/logs/sc_rna_annotated_minmi0.0_top2000/seed_42/multitask/DeepTAN_20250407160504_ZwW8D/"+"version_0"
_logs_local = "/mnt/hdd2/homext/wuch/xn2p/run/logs"
_tsb_local = collect_tensorboard_events(_logs_local)

Found 25 checkpoints.



In [3]:
_logs_hpc = "/mnt/hdd2/homext/wuch/xn2p/run/logs_hpc"
_tsb_hpc = collect_tensorboard_events(_logs_hpc)

Found 198 checkpoints.



In [4]:
_tsb = pl.concat([_tsb_local, _tsb_hpc], how="diagonal", rechunk=True)

In [5]:
print(_tsb.columns)

['ckpt_path', 'log_name', 'task', 'seed', 'data', 'val/loss', 'val/loss_unweighted', 'val/recon_MSE', 'val/recon_MAE', 'val/recon_PCC', 'val/label_F1_weighted', 'val/label_F1_macro', 'val/label_F1_micro', 'val/label_AUROC', 'val/label_Accuracy', 'val/label_Precision', 'val/label_Recall', 'test/recon_MSE', 'test/recon_MAE', 'test/recon_PCC', 'test/label_F1_weighted', 'test/label_F1_macro', 'test/label_F1_micro', 'test/label_AUROC', 'test/label_Accuracy', 'test/label_Precision', 'test/label_Recall', 'test/loss', 'test/loss_unweighted', 'test/recon_RMSE', 'val/recon_RMSE']


In [6]:
print(_tsb)

shape: (223, 31)
┌────────────┬───────────┬───────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ ckpt_path  ┆ log_name  ┆ task      ┆ seed    ┆ … ┆ test/loss ┆ test/loss ┆ test/reco ┆ val/recon │
│ ---        ┆ ---       ┆ ---       ┆ ---     ┆   ┆ ---       ┆ _unweight ┆ n_RMSE    ┆ _RMSE     │
│ str        ┆ str       ┆ str       ┆ str     ┆   ┆ f64       ┆ ed        ┆ ---       ┆ ---       │
│            ┆           ┆           ┆         ┆   ┆           ┆ ---       ┆ f64       ┆ f64       │
│            ┆           ┆           ┆         ┆   ┆           ┆ f64       ┆           ┆           │
╞════════════╪═══════════╪═══════════╪═════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ /mnt/hdd2/ ┆ DeepTAN_2 ┆ multitask ┆ seed_42 ┆ … ┆ null      ┆ null      ┆ null      ┆ null      │
│ homext/wuc ┆ 025033018 ┆           ┆         ┆   ┆           ┆           ┆           ┆           │
│ h/xn2p/run ┆ 5410_lp5o ┆           ┆         ┆   ┆           ┆          

In [7]:
_tsb.write_csv(".collected_logs/logs_tsb.csv")
_tsb.write_parquet(".collected_logs/logs_tsb.parquet")

In [8]:
best_ckpts = _tsb.group_by(["data", "task", "seed"]).agg([pl.col("val/loss").min()]).join(_tsb, on=["data", "task", "seed", "val/loss"], how="left")
best_ckpts = best_ckpts.sort(by=["data", "task", "seed"])
print(best_ckpts)

shape: (21, 31)
┌────────────┬────────────┬─────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ data       ┆ task       ┆ seed    ┆ val/loss ┆ … ┆ test/loss ┆ test/loss ┆ test/reco ┆ val/recon │
│ ---        ┆ ---        ┆ ---     ┆ ---      ┆   ┆ ---       ┆ _unweight ┆ n_RMSE    ┆ _RMSE     │
│ str        ┆ str        ┆ str     ┆ f64      ┆   ┆ f64       ┆ ed        ┆ ---       ┆ ---       │
│            ┆            ┆         ┆          ┆   ┆           ┆ ---       ┆ f64       ┆ f64       │
│            ┆            ┆         ┆          ┆   ┆           ┆ f64       ┆           ┆           │
╞════════════╪════════════╪═════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ sc_multiom ┆ multitask  ┆ seed_42 ┆ 0.308697 ┆ … ┆ 0.206165  ┆ 0.117851  ┆ null      ┆ null      │
│ e          ┆            ┆         ┆          ┆   ┆           ┆           ┆           ┆           │
│ sc_multiom ┆ focus_labe ┆ seed_42 ┆ 0.026796 ┆ … ┆ 0.022345  ┆ 1.168914  

In [9]:
best_ckpts.write_csv(".collected_logs/best_ckpts.csv")
# best_ckpts.write_excel(".collected_logs/best_ckpts.xlsx")
best_ckpts.write_parquet(".collected_logs/best_ckpts.parquet")

### Best in seed_42

In [10]:
import polars as pl

In [11]:
best_ckpts = pl.read_parquet(".collected_logs/best_ckpts.parquet")

In [12]:
best_ckpts_seed_42 = best_ckpts.filter(pl.col("seed") == "seed_42")
# print(best_ckpts_seed_42)
best_ckpts_seed_42.write_parquet(".collected_logs/best_ckpts_seed_42.parquet")
best_ckpts_seed_42.write_csv(".collected_logs/best_ckpts_seed_42.csv")
# best_ckpts_seed_42.write_excel(".collected_logs/best_ckpts_seed_42.xlsx")

### Best in seeds

In [13]:
import polars as pl

In [15]:
best_ckpts = pl.read_parquet(".collected_logs/best_ckpts.parquet")

In [16]:
best_seeds = best_ckpts.group_by(["data", "task"]).agg([pl.col("val/loss").min()]).join(best_ckpts, on=["data", "task", "val/loss"], how="left").sort(by=["data", "task"])

In [17]:
print(best_seeds)

shape: (9, 31)
┌────────────┬────────────┬──────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ data       ┆ task       ┆ val/loss ┆ seed    ┆ … ┆ test/loss ┆ test/loss ┆ test/reco ┆ val/recon │
│ ---        ┆ ---        ┆ ---      ┆ ---     ┆   ┆ ---       ┆ _unweight ┆ n_RMSE    ┆ _RMSE     │
│ str        ┆ str        ┆ f64      ┆ str     ┆   ┆ f64       ┆ ed        ┆ ---       ┆ ---       │
│            ┆            ┆          ┆         ┆   ┆           ┆ ---       ┆ f64       ┆ f64       │
│            ┆            ┆          ┆         ┆   ┆           ┆ f64       ┆           ┆           │
╞════════════╪════════════╪══════════╪═════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ sc_multiom ┆ multitask  ┆ 0.308697 ┆ seed_42 ┆ … ┆ 0.206165  ┆ 0.117851  ┆ null      ┆ null      │
│ e          ┆            ┆          ┆         ┆   ┆           ┆           ┆           ┆           │
│ sc_multiom ┆ focus_labe ┆ 0.026796 ┆ seed_42 ┆ … ┆ 0.022345  ┆ 1.168914  ┆

In [18]:
best_seeds.write_csv(".collected_logs/best_seeds.csv")
# best_seeds.write_excel(".collected_logs/best_seeds.xlsx")
best_seeds.write_parquet(".collected_logs/best_seeds.parquet")